# 9.3 Publication Text Analysis - step 3: TF-IDF

This notebook:

1. Runs simple TF-IDF vectorizer on the publication text (title + abstract)
2. Investigates the output, and effects of parameters
3. Investigates the prevalence of terms among high and low energy labs

In [1]:
# Set up
import pandas as pd
import numpy as np
import sys
import re
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from scipy.sparse import csr_matrix
from scipy.stats import spearmanr

In [2]:
# Load data
publications = pd.read_csv(
    config.PUBLICATON_DATA /
    "3_Clean" /
    "publications_unique.csv"
)

lab_data = pd.read_csv(
    config.PUBLICATON_DATA /
    "3_Clean" /
    "lab_level_subject_shares.csv"
)

pub_lab_map = pd.read_csv(
    config.PUBLICATON_DATA / 
    "3_Clean" / 
    "pub_lab_mapping.csv"
)

## (1) TfidfVectorizer baseline - run, check output, inspect issues

In [3]:
# Configure the vectorizer
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.95,
    min_df = 2,
)

# fit_transform: learn the vocabulary from the corpus (fit), then convert
# every document into its TF-IDF vector (transform) - one sparse matrix,
# rows = documents, columns = vocabulary terms
X = tfidf.fit_transform(publications["text_final"])

print(f"Matrix shape: {X.shape[0]:,} documents x {X.shape[1]:,} vocabulary terms")

Matrix shape: 8,014 documents x 94,615 vocabulary terms


In [4]:
# Get the vocabulary terms from the vectorizer
vocab = tfidf.get_feature_names_out()
print(f"Vocabulary size: {len(vocab):,}")

# Check a random sample of terms
rng = np.random.default_rng(0)
print("\nRandom sample of 25 terms:")
print(sorted(rng.choice(vocab, size=25, replace=False)))

Vocabulary size: 94,615

Random sample of 25 terms:
['07 text', '205', 'acid diethylene', 'antigenic', 'cloud forest', 'different nature', 'disc evolution', 'egg rejection', 'genes plant', 'landscape structure', 'lec', 'malacochersus tornieri', 'mechanism maintain', 'ncc dephosphorylation', 'observe significant', 'oelv trimethylamine', 'overcome problems', 'performed automated', 'proteins peptides', 'second systematic', 'segment specific', 'sorption', 'tensor integral', 'transition emt', 'vapour pressure']


## (2) Inspect the vocabulary

In [5]:
# Document frequency per term - how many documents does each term appear in?
doc_freq = np.asarray((X > 0).sum(axis=0)).ravel()

n_at_floor = (doc_freq == 2).sum()
print(f"\nTerms right at the min_df floor (doc frequency = 2): {n_at_floor:,} ({n_at_floor / len(vocab):.1%} of vocabulary)")

print("\n25 random terms at the floor (doc frequency = 2):")
floor_terms = vocab[doc_freq == 2]
print(sorted(rng.choice(floor_terms, size=25, replace=False)))


Terms right at the min_df floor (doc frequency = 2): 50,264 (53.1% of vocabulary)

25 random terms at the floor (doc frequency = 2):
['calving drafts', 'electroreductive', 'formulation qt', 'free isomers', 'functional metabolic', 'galaxies suggests', 'hhh', 'increasing application', 'invasive nature', 'literature reports', 'mitogen activated', 'nhej homologous', 'p_t distribution', 'plants plants', 'pm 31', 'predispose pathologies', 'proliferating nspcs', 'regions differentially', 'samples cp', 'site selectivity', 'smooth particle', 'strategy reduce', 'unconventional magnetism', 'vitro analysis', 'zero charged']


## (3) Check how much of vocab contains a digit

In [6]:
has_digit = np.array([bool(re.search(r"\d", term)) for term in vocab])
print(f"Vocabulary terms containing a digit: {has_digit.sum():,} ({has_digit.mean():.1%} of vocabulary)")

print("Sample of digit-containing terms:")
print(sorted(rng.choice(vocab[has_digit], size=15, replace=False)))

Vocabulary terms containing a digit: 6,222 (6.6% of vocabulary)
Sample of digit-containing terms:
['10 31', '100 ns', '150 µg', '2016 corresponding', '302 01', '38 cells', '46 patients', '54 species', 'calculated bmdl05', 'exceeding 10', 'material 27', 'mehtm 5cx', 'p450 cyp', 'pm3 haplotype', 'q1']


## (4) Investigation of min_df

In [7]:
# Investigate the effect of min_df on the vocabulary size and presence of specific terms
test_terms = ["fume cupboard", "cryostat", "catalysis", "cryogenic", "spectrometer", "laser"]  # some terms to check
min_df_values = [1, 2, 3, 5, 10]

for md in min_df_values:
    tfidf_test = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_df=0.95,
        min_df=md,
    )
    tfidf_test.fit(publications["text_final"])
    vocab_test = set(tfidf_test.get_feature_names_out())

    present = [t for t in test_terms if t in vocab_test]
    print(f"min_df={md}: vocab size = {len(vocab_test):,}, "
          f"test terms present = {present}")

min_df=1: vocab size = 539,443, test terms present = ['cryostat', 'catalysis', 'cryogenic', 'spectrometer', 'laser']
min_df=2: vocab size = 94,615, test terms present = ['catalysis', 'cryogenic', 'spectrometer', 'laser']
min_df=3: vocab size = 44,351, test terms present = ['catalysis', 'cryogenic', 'spectrometer', 'laser']
min_df=5: vocab size = 22,129, test terms present = ['catalysis', 'spectrometer', 'laser']
min_df=10: vocab size = 10,602, test terms present = ['catalysis', 'spectrometer', 'laser']


There is a trade-off in the presence of specific terms vs. the vocabulary size. For now we choose min_df = 3 as this is where we lose some terms.

In [8]:
# Configure the vectorizer
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.95,
    min_df = 3,
)
X = tfidf.fit_transform(publications["text_final"])

## (5) Split labs into high/low energy

In [9]:
# Split labs on median annual electricity consumption
median_energy = lab_data["annual_electricity_total"].median()

lab_data["energy_tier"] = np.where(
    lab_data["annual_electricity_total"] >= median_energy, "high", "low"
)

print(f"Median annual_electricity_total: {median_energy:,.0f}")
print(lab_data["energy_tier"].value_counts())

Median annual_electricity_total: 14,035
energy_tier
high    48
low     47
Name: count, dtype: int64


## (6) Build per-energy tier word-count aggregates

In [10]:
# Merge the energy tier info with the pub-lab mapping and publications
pub_lab_tier = pub_lab_map.merge(lab_data[["labgroupid", "energy_tier"]], on="labgroupid", how="left")
pub_lab_tier = pub_lab_tier.merge(publications[["pub_id", "text_final"]], on="pub_id", how="left")

print(f"Pub-lab rows with a tier assigned: {pub_lab_tier['energy_tier'].notna().sum():,} / {len(pub_lab_tier):,}")
print(pub_lab_tier["energy_tier"].value_counts())

Pub-lab rows with a tier assigned: 8,435 / 8,435
energy_tier
high    4402
low     4033
Name: count, dtype: int64


In [11]:
# Configure the CountVectorizer with the same vocabulary and settings as the TF-IDF vectorizer
count_vec = CountVectorizer(
    vocabulary=tfidf.vocabulary_,
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
)

# Split the publications into high and low energy tiers
high_text = pub_lab_tier.loc[pub_lab_tier["energy_tier"] == "high", "text_final"]
low_text = pub_lab_tier.loc[pub_lab_tier["energy_tier"] == "low", "text_final"]

# Compute the total in-vocabulary word occurrences for each tier
high_counts = np.asarray(count_vec.transform(high_text).sum(axis=0)).ravel()
low_counts = np.asarray(count_vec.transform(low_text).sum(axis=0)).ravel()

print(f"Total in-vocabulary word occurrences - high tier: {high_counts.sum():,.0f}")
print(f"Total in-vocabulary word occurrences - low tier: {low_counts.sum():,.0f}")

Total in-vocabulary word occurrences - high tier: 514,036
Total in-vocabulary word occurrences - low tier: 485,062


## (7) Which terms are more typical of high vs. low energy tier labs?

In [12]:
# We use the log-odds ratio with an informative Dirichlet prior
# This accounts for overall frequency of words in corpus, and avoids 0 count issues

# Combined counts
alpha = high_counts + low_counts
alpha_0 = alpha.sum()

# Total word counts for each tier
n_high = high_counts.sum()
n_low = low_counts.sum()

# Compute the log-odds ratio for each term (with )
log_odds_high = np.log((high_counts + alpha) / (n_high + alpha_0 - high_counts - alpha))
log_odds_low = np.log((low_counts + alpha) / (n_low + alpha_0 - low_counts - alpha))

# Compute the z-scores for each term
delta = log_odds_high - log_odds_low
variance = 1 / (high_counts + alpha) + 1 / (low_counts + alpha)
z_scores = delta / np.sqrt(variance)

# Create DF of results
terms = tfidf.get_feature_names_out()
results = pd.DataFrame({
    "term": terms,
    "high_count": high_counts,
    "low_count": low_counts,
    "z_score": z_scores,
})

In [13]:
# Check top 20 terms distinctive of high energy labs
print("Top 20 terms distinctive of high energy labs:")
print(results.sort_values("z_score", ascending=False).head(20).to_string(index=False))

Top 20 terms distinctive of high energy labs:
               term  high_count  low_count   z_score
                mak        1385          0 20.502064
              value        1605        150 19.144348
          mak value        1106          0 18.316125
              cells        1800        307 17.888747
               mice         953         14 16.633350
                 m3         733          0 14.905690
               acid         698         27 13.727536
                ebv         628          7 13.569572
          compounds         687         31 13.486162
                 mg         658         25 13.341762
            protein         921        151 12.925242
           toxicity         547         11 12.495096
         commission         521          2 12.493011
             german         627         52 12.152529
               cell        1462        486 12.001437
               rats         458          2 11.703485
value documentation         426          0 11.359955


In [14]:
# Check top 20 terms distinctive of low energy labs
print("\nTop 20 terms distinctive of low energy labs:")
print(results.sort_values("z_score", ascending=True).head(20).to_string(index=False))


Top 20 terms distinctive of low energy labs:
                 term  high_count  low_count    z_score
               proton         275       1354 -15.990675
                  tev         261       1196 -14.677102
           collisions         205       1019 -13.907896
         neuromorphic           0        539 -13.512878
               13 tev          58        595 -12.393510
                 sqrt         105        688 -12.295791
                 mass         578       1487 -12.284891
             galaxies           1        437 -12.128417
        proton proton          99        643 -11.862392
                calls          10        424 -11.604179
              sqrt 13          23        452 -11.537022
    proton collisions          59        526 -11.407489
               cancer         369       1074 -11.310763
      collisions sqrt          68        522 -11.067875
                   95         131        645 -11.027388
               galaxy           0        359 -11.026186
  

We also want to know whether these terms are coming from just one lab's publications, or from publications across several labs.

In [15]:
# Term presence (0/1) per pub-lab row
X_bin = (count_vec.transform(pub_lab_tier["text_final"]) > 0).astype(int)

# Collapse rows to labs: a lab "uses" a term if any of its pubs do
labs_unique = pub_lab_tier["labgroupid"].unique()
lab_idx = {lab: i for i, lab in enumerate(labs_unique)}
row_lab_idx = pub_lab_tier["labgroupid"].map(lab_idx).values

# Build a (labs x rows) indicator matrix
indicator = csr_matrix(
    (np.ones(len(row_lab_idx)), (row_lab_idx, np.arange(len(row_lab_idx)))),
    shape=(len(labs_unique), len(pub_lab_tier)),
)
lab_term_presence = (indicator @ X_bin) > 0  # labs x terms

# Count number of high and low energy labs that use each term
lab_tier_lookup = pub_lab_tier.drop_duplicates("labgroupid").set_index("labgroupid")["energy_tier"]
lab_is_high = np.array([lab_tier_lookup[lab] == "high" for lab in labs_unique])

n_high_labs = np.asarray(lab_term_presence[lab_is_high].sum(axis=0)).ravel()
n_low_labs = np.asarray(lab_term_presence[~lab_is_high].sum(axis=0)).ravel()

results["n_high_labs"] = n_high_labs
results["n_low_labs"] = n_low_labs

In [16]:
print("Top 20 terms distinctive of high-energy labs (with distinct-lab counts):")
print(results.sort_values("z_score", ascending=False).head(20).to_string(index=False))

Top 20 terms distinctive of high-energy labs (with distinct-lab counts):
               term  high_count  low_count   z_score  n_high_labs  n_low_labs
                mak        1385          0 20.502064            1           0
              value        1605        150 19.144348           23          25
          mak value        1106          0 18.316125            1           0
              cells        1800        307 17.888747           43          16
               mice         953         14 16.633350           27           1
                 m3         733          0 14.905690            1           0
               acid         698         27 13.727536           31           8
                ebv         628          7 13.569572            3           1
          compounds         687         31 13.486162           21           8
                 mg         658         25 13.341762           16           8
            protein         921        151 12.925242           37    

In [17]:
print("\nTop 20 terms distinctive of low-energy labs (with distinct-lab counts):")
print(results.sort_values("z_score", ascending=True).head(20).to_string(index=False))


Top 20 terms distinctive of low-energy labs (with distinct-lab counts):
                 term  high_count  low_count    z_score  n_high_labs  n_low_labs
               proton         275       1354 -15.990675           12           6
                  tev         261       1196 -14.677102            2           6
           collisions         205       1019 -13.907896            2           9
         neuromorphic           0        539 -13.512878            0           2
               13 tev          58        595 -12.393510            2           5
                 sqrt         105        688 -12.295791            2           5
                 mass         578       1487 -12.284891           28          22
             galaxies           1        437 -12.128417            1           2
        proton proton          99        643 -11.862392            2           3
                calls          10        424 -11.604179            7          10
              sqrt 13          23   

We see that some terms come from exactly 1 lab group's publications - we will need to deal with this in our analysis.

We also want to check whether we get similar results with the log-likelihood ratio.

In [18]:
# Compute the G-test statistic (log-likelihood ratio) for each term

# Observed counts for each term in high and low tiers
a = high_counts.astype(float)  # observed count of term in high tier
b = low_counts.astype(float)   # observed count of term in low tier
a_not = n_high - a             # observed count of everything else in high tier
b_not = n_low - b              # observed count of everything else in low tier

# Compute expected counts
term_total = a + b
E_a = n_high * term_total / (n_high + n_low)       # expected count of term in high tier, if equally likely
E_b = n_low * term_total / (n_high + n_low)         # expected count of term in low tier, if equally likely
E_a_not = n_high - E_a
E_b_not = n_low - E_b

# Compute the G-test statistic for each term
def xlogx_ratio(observed, expected):
    # observed * log(observed / expected), with observed=0 contributing 0
    # (the standard convention for this test - avoids log(0))
    with np.errstate(divide="ignore", invalid="ignore"):
        val = observed * np.log(observed / expected)
    return np.where(observed > 0, val, 0.0)

G2 = 2 * (
    xlogx_ratio(a, E_a) + xlogx_ratio(b, E_b)
    + xlogx_ratio(a_not, E_a_not) + xlogx_ratio(b_not, E_b_not)
)

# Sign: term is over-represented in high tier (+) or low tier (-)
sign = np.where(a >= E_a, 1, -1)
results["g2_score"] = sign * G2

In [19]:
# Check top 20 terms distinctive of high energy labs by G-test
print("Top 20 terms distinctive of HIGH-energy labs by G-test:")
print(results.sort_values("g2_score", ascending=False).head(20).to_string(index=False))

Top 20 terms distinctive of HIGH-energy labs by G-test:
               term  high_count  low_count   z_score  n_high_labs  n_low_labs    g2_score
                mak        1385          0 20.502064            1           0 1842.644214
          mak value        1106          0 18.316125            1           0 1471.162355
              value        1605        150 19.144348           23          25 1327.313014
               mice         953         14 16.633350           27           1 1141.334476
              cells        1800        307 17.888747           43          16 1088.541893
                 m3         733          0 14.905690            1           0  974.752157
                ebv         628          7 13.569572            3           1  768.136298
               acid         698         27 13.727536           31           8  736.507937
          compounds         687         31 13.486162           21           8  702.833235
                 mg         658         25 1

In [20]:
# Check top 20 terms distinctive of low energy labs by G-test
print("\nTop 20 terms distinctive of LOW-energy labs by G-test:")
print(results.sort_values("g2_score", ascending=True).head(20).to_string(index=False))


Top 20 terms distinctive of LOW-energy labs by G-test:
                 term  high_count  low_count    z_score  n_high_labs  n_low_labs    g2_score
               proton         275       1354 -15.990675           12           6 -844.376200
         neuromorphic           0        539 -13.512878            0           2 -779.245416
                  tev         261       1196 -14.677102            2           6 -706.448329
           collisions         205       1019 -13.907896            2           9 -639.616587
             galaxies           1        437 -12.128417            1           2 -618.900300
               13 tev          58        595 -12.393510            2           5 -545.724076
                calls          10        424 -11.604179            7          10 -531.041160
               galaxy           0        359 -11.026186            0           3 -518.946431
                 sqrt         105        688 -12.295791            2           5 -514.159953
              

Now we want to compare the two methods directly.

In [21]:
# Rank correlation across the whole vocabulary
corr, _ = spearmanr(results["z_score"], results["g2_score"])
print(f"Spearman rank correlation between z_score and g2_score (all {len(results):,} terms): {corr:.3f}")

# Overlap in the top 20 terms distinctive of high-energy labs
top20_zscore_high = set(results.sort_values("z_score", ascending=False).head(20)["term"])
top20_g2_high = set(results.sort_values("g2_score", ascending=False).head(20)["term"])
overlap_high = top20_zscore_high & top20_g2_high
print(f"\nTop-20 high-tier term overlap: {len(overlap_high)}/20")
print(f"In z_score top 20 but not g2 top 20: {top20_zscore_high - top20_g2_high}")
print(f"In g2 top 20 but not z_score top 20: {top20_g2_high - top20_zscore_high}")

# Overlap in the top 20 terms distinctive of low-energy labs
top20_zscore_low = set(results.sort_values("z_score", ascending=True).head(20)["term"])
top20_g2_low = set(results.sort_values("g2_score", ascending=True).head(20)["term"])
overlap_low = top20_zscore_low & top20_g2_low
print(f"\nTop-20 low-tier term overlap: {len(overlap_low)}/20")
print(f"In z_score top 20 but not g2 top 20: {top20_zscore_low - top20_g2_low}")
print(f"In g2 top 20 but not z_score top 20: {top20_g2_low - top20_zscore_low}")

Spearman rank correlation between z_score and g2_score (all 44,351 terms): 0.998

Top-20 high-tier term overlap: 19/20
In z_score top 20 but not g2 top 20: {'cell'}
In g2 top 20 but not z_score top 20: {'ml m3'}

Top-20 low-tier term overlap: 19/20
In z_score top 20 but not g2 top 20: {'13'}
In g2 top 20 but not z_score top 20: {'stellar'}


Overall findings:

1. Terms to do with particle physics seem particularly prevalent among low-energy labs (perhaps because they are purely computational).
2. Terms to do with wet labs/biology seem particularly prevalent among high-energy labs - this makes sense that these labs are actually doing the resource-intensive research.
3. Terms to do with "MAK" and "value documentation" are also particularly prevalent among high-energy labs - but these are from only 1/2 labs. We also identify some other terms that are from only 1/2 labs - we will need to account for this later on.